## 01 Bronze Layer: Raw Mock Data Ingestion
Generates all four mock Bronze tables with intentional quality issues and writes
them as Delta tables into the project catalog/schema.

**Tables created:**
- `bronze_vendors`
- `bronze_purchase_orders`
- `bronze_line_items`
- `bronze_spend_taxonomy`

Data is intentionally messy *(nulls, wrong types, inconsistent formatting)* to simulate real-world procurement data that needs cleaning in Silver.

### Imports

In [0]:
import logging
import random
from datetime import date, datetime, timedelta

from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType,
    StringType,
    StructField,
    StructType,
)

# Logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("procurement")

# Catalog / schema constants
CATALOG = "healthcare_procurement"
SCHEMA  = "procurement_analytics"

log.info("Imports complete. Catalog=%s  Schema=%s", CATALOG, SCHEMA)

20:41:54  INFO  Imports complete. Catalog=healthcare_procurement  Schema=procurement_analytics


### Create Catalog and Schema

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")
log.info("Catalog and schema ready: %s.%s", CATALOG, SCHEMA)

20:41:57  INFO  Catalog and schema ready: healthcare_procurement.procurement_analytics


### Helper Functions

In [0]:
# Reusable utilities
def write_table(df, table_name: str, mode: str = "overwrite") -> None:
    # Write a Spark DataFrame as a Delta table in the active schema.
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    df.write.format("delta").mode(mode).saveAsTable(full_name)
    log.info("Written -> %s  (%d rows)", full_name, df.count())

def null_report(df, label: str) -> None:
    #Print a count of null values for every column in the DataFrame.
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
    ])
    print(f"\nNull report -- {label}")
    null_counts.show()

log.info("Helper functions defined.")

20:41:57  INFO  Helper functions defined.


### Bronze: `bronze_vendors`

**Intentional quality issues:**
- ~10% nulls in `city` and `country`
- Inconsistent casing/spelling in `vendor_name`
- Inconsistent `payment_terms` formatting (net30, NET30, Net 30)
- Inconsistent `vendor_category` casing

In [0]:
random.seed(42)

vendor_data_clean = [
    ("V001", "Medline Industries",         "Chicago",       "USA",         "Net 30", "Medical Supplies"),
    ("V002", "Cardinal Health",            "Dublin",        "USA",         "Net 60", "Medical Supplies"),
    ("V003", "Johnson & Johnson",          "New Brunswick", "USA",         "Net 30", "Medical Equipment"),
    ("V004", "Baxter International",       "Deerfield",     "USA",         "Net 45", "Pharmacy"),
    ("V005", "McKesson Corporation",       "Irving",        "USA",         "Net 30", "Medical Supplies"),
    ("V006", "Stryker Corporation",        "Kalamazoo",     "USA",         "Net 60", "Medical Equipment"),
    ("V007", "BD (Becton Dickinson)",      "Franklin Lakes","USA",         "Net 30", "Medical Supplies"),
    ("V008", "Siemens Healthineers",       "Erlangen",      "Germany",     "Net 60", "Medical Equipment"),
    ("V009", "Philips Healthcare",         "Amsterdam",     "Netherlands", "Net 45", "Medical Equipment"),
    ("V010", "3M Health Care",             "Maplewood",     "USA",         "Net 30", "Medical Supplies"),
    ("V011", "Owens & Minor",              "Mechanicsville","USA",         "Net 30", "Medical Supplies"),
    ("V012", "Henry Schein",               "Melville",      "USA",         "Net 45", "Dental & Surgical"),
    ("V013", "Fresenius Kabi",             "Bad Homburg",   "Germany",     "Net 60", "Pharmacy"),
    ("V014", "Olympus Medical",            "Tokyo",         "Japan",       "Net 60", "Medical Equipment"),
    ("V015", "Zimmer Biomet",              "Warsaw",        "USA",         "Net 45", "Medical Equipment"),
    ("V016", "Staples Business Advantage", "Framingham",    "USA",         "Net 30", "Administrative"),
    ("V017", "Cintas Corporation",         "Cincinnati",    "USA",         "Net 30", "Facilities"),
    ("V018", "Aramark",                    "Philadelphia",  "USA",         "Net 30", "Facilities"),
    ("V019", "Abbott Laboratories",        "North Chicago", "USA",         "Net 30", "Pharmacy"),
    ("V020", "GE Healthcare",              "Chicago",       "USA",         "Net 60", "Medical Equipment"),
]

# Introduces small, realistic text variations to vendor names
def corrupt_vendor_name(name):
    r = random.random()
    if r < 0.15: return name.lower()          # lowercase version
    elif r < 0.25: return name.upper()        # uppercase version
    elif r < 0.35: return name.replace("&", "and")  # replace symbol
    elif r < 0.40: return name.replace(" ", "  ")   # double spaces
    return name                                # unchanged

# Randomizes payment term formats to simulate messy real-world data
def corrupt_payment_terms(terms):
    variations = {
        "Net 30": ["net30", "NET30", "n/30", "Net30", "NET 30", "Net 30"],
        "Net 60": ["net60", "NET60", "n/60", "Net60", "NET 60", "Net 60"],
        "Net 45": ["net45", "NET45", "n/45", "Net45", "NET 45", "Net 45"],
    }
    return random.choice(variations.get(terms, [terms]))

# Build Bronze vendor rows with intentional corruption + null injection
vendors_rows = []
for v in vendor_data_clean:
    vid, name, city, country, terms, cat = v
    vendors_rows.append((
        vid,
        corrupt_vendor_name(name),                         # messy vendor names
        None if random.random() < 0.10 else city,          # 10% null city
        None if random.random() < 0.10 else country,       # 10% null country
        corrupt_payment_terms(terms),                      # messy payment terms
        cat if random.random() > 0.15 else cat.upper(),    # 15% uppercase category
    ))

# Schema for Bronze vendor table (raw, semi-structured, nullable)
vendor_schema = StructType([
    StructField("vendor_id",       StringType(), False),
    StructField("vendor_name",     StringType(), True),
    StructField("city",            StringType(), True),
    StructField("country",         StringType(), True),
    StructField("payment_terms",   StringType(), True),
    StructField("vendor_category", StringType(), True),
])

# Create Bronze DataFrame and write to Unity Catalog
df_vendors_bronze = spark.createDataFrame(vendors_rows, schema=vendor_schema)
write_table(df_vendors_bronze, "bronze_vendors")
null_report(df_vendors_bronze, "bronze_vendors")
df_vendors_bronze.show(5, truncate=False)

20:42:01  INFO  Written -> healthcare_procurement.procurement_analytics.bronze_vendors  (20 rows)



Null report -- bronze_vendors
+---------+-----------+----+-------+-------------+---------------+
|vendor_id|vendor_name|city|country|payment_terms|vendor_category|
+---------+-----------+----+-------+-------------+---------------+
|        0|          0|   4|      0|            0|              0|
+---------+-----------+----+-------+-------------+---------------+

+---------+--------------------+-------------+-------+-------------+-----------------+
|vendor_id|vendor_name         |city         |country|payment_terms|vendor_category  |
+---------+--------------------+-------------+-------+-------------+-----------------+
|V001     |Medline Industries  |NULL         |USA    |NET30        |MEDICAL SUPPLIES |
|V002     |cardinal health     |Dublin       |USA    |NET 60       |Medical Supplies |
|V003     |johnson & johnson   |New Brunswick|USA    |net30        |Medical Equipment|
|V004     |Baxter International|Deerfield    |USA    |Net45        |Pharmacy         |
|V005     |McKesson Corp

### Bronze: `bronze_purchase_orders`

**Intentional quality issues:**
- `po_date` and `paid_date` stored as STRING in mixed formats
- `paid_date` ~15% null — Pending/Rejected POs have no payment date by design
- ~5% nulls in `vendor_id`
- `total_amount` stored as STRING with currency symbols
- Inconsistent `status` labels (approved, APPROVED, aprvd)
- Mixed casing in `department_id`

`po_date` = when the PO was raised. `paid_date` = when the invoice was paid.
The gap between them (Days to Pay) is used in Gold to flag late payments.

In [0]:
random.seed(99)

DEPARTMENTS = ["ICU", "Emergency", "Radiology", "Surgery", "Pharmacy",
               "Cardiology", "Oncology", "Pediatrics", "Administration", "Facilities"]
STATUS_VARIANTS = {
    "Approved": ["Approved", "approved", "APPROVED", "aprvd", "Aprvd"],
    "Pending":  ["Pending",  "pending",  "PENDING",  "pndng", "In Review"],
    "Rejected": ["Rejected", "rejected", "REJECTED", "Reject", "REJECT"],
}

# Generate a random date string in mixed formats
def random_date_str(start_year=2023, end_year=2024):
    start = date(start_year, 1, 1)
    end   = date(end_year, 12, 31)
    d     = start + timedelta(days=random.randint(0, (end - start).days))
    r = random.random()
    if r < 0.30:   return d.strftime("%m/%d/%Y")   # MM/DD/YYYY
    elif r < 0.50: return d.strftime("%d-%m-%Y")   # DD-MM-YYYY (ambiguous)
    else:          return d.strftime("%Y-%m-%d")   # ISO (correct)

# Introduce formatting inconsistencies into numeric amounts
def corrupt_amount(amount):
    r = random.random()
    if r < 0.30:   return f"${amount:,.2f}"   # currency format
    elif r < 0.45: return f"{amount:.2f}"     # decimal string
    else:          return str(round(amount))  # integer-like

# Generate paid_date based on status + random delays + mixed formats
def generate_paid_date(po_date_str, true_status):
    # Most Pending/Rejected have no paid_date
    if true_status in ("Pending", "Rejected") and random.random() < 0.85:
        return None
    # Small chance Approved still has no paid_date
    if true_status == "Approved" and random.random() < 0.05:
        return None
    # Parse po_date using multiple possible formats
    for fmt in ("%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y"):
        try:
            base = datetime.strptime(po_date_str, fmt).date()
            break
        except ValueError:
            continue
    # Paid date is 15–75 days after PO date
    paid = base + timedelta(days=random.randint(15, 75))
    # Return paid_date in mixed formats
    r = random.random()
    if r < 0.30:   return paid.strftime("%m/%d/%Y")
    elif r < 0.50: return paid.strftime("%d-%m-%Y")
    else:          return paid.strftime("%Y-%m-%d")

# Extract vendor IDs from cleaned vendor dataset
vendor_ids = [v[0] for v in vendor_data_clean]

# Build Bronze purchase order rows with intentional corruption + nulls
po_rows = []
for i in range(1, 501):
    po_date     = random_date_str()
    true_status = random.choices(["Approved", "Pending", "Rejected"], weights=[0.70, 0.20, 0.10])[0]
    dept        = random.choice(DEPARTMENTS)
    po_rows.append((
        f"PO-{i:04d}",                                   # PO ID
        po_date,                                         # messy date formats
        generate_paid_date(po_date, true_status),        # messy paid_date
        None if random.random() < 0.05 else random.choice(vendor_ids),  # 5% null vendor_id
        dept.lower() if random.random() < 0.15 else      # lowercase variation
        (dept + " DEPT" if random.random() < 0.10 else dept),  # suffix variation
        corrupt_amount(round(random.uniform(200, 50000), 2)),  # messy amount formats
        random.choice(STATUS_VARIANTS[true_status]),     # inconsistent status label
    ))

# Bronze schema: intentionally loose, string-heavy, nullable
po_schema = StructType([
    StructField("po_id",         StringType(), False),
    StructField("po_date",       StringType(), True),  # intentionally STRING
    StructField("paid_date",     StringType(), True),  # intentionally STRING, ~15% null
    StructField("vendor_id",     StringType(), True),
    StructField("department_id", StringType(), True),
    StructField("total_amount",  StringType(), True),  # intentionally STRING
    StructField("status",        StringType(), True),
])

# Create Bronze DataFrame and write to Unity Catalog
df_po_bronze = spark.createDataFrame(po_rows, schema=po_schema)
write_table(df_po_bronze, "bronze_purchase_orders")
null_report(df_po_bronze, "bronze_purchase_orders")
df_po_bronze.show(5, truncate=False)

20:42:05  INFO  Written -> healthcare_procurement.procurement_analytics.bronze_purchase_orders  (500 rows)



Null report -- bronze_purchase_orders
+-----+-------+---------+---------+-------------+------------+------+
|po_id|po_date|paid_date|vendor_id|department_id|total_amount|status|
+-----+-------+---------+---------+-------------+------------+------+
|    0|      0|      149|       19|            0|           0|     0|
+-----+-------+---------+---------+-------------+------------+------+

+-------+----------+----------+---------+-------------+------------+---------+
|po_id  |po_date   |paid_date |vendor_id|department_id|total_amount|status   |
+-------+----------+----------+---------+-------------+------------+---------+
|PO-0001|18-02-2024|04/21/2024|V017     |Surgery      |46931       |aprvd    |
|PO-0002|2024-09-19|NULL      |V007     |Cardiology   |42528.37    |In Review|
|PO-0003|2024-11-24|NULL      |V014     |Oncology     |26809       |approved |
|PO-0004|12-09-2024|2024-11-24|V015     |Radiology    |47207       |Approved |
|PO-0005|04/23/2023|2023-06-04|V020     |oncology     |16

### Bronze: `bronze_line_items`

**Intentional quality issues:**
- `unit_price` stored as STRING with occasional "$" prefix or whitespace
- ~3-5% of `quantity` values are zero or negative
- ~2% null `item_description`
- Free-text `item_description` in inconsistent casing

In [0]:
random.seed(7)

ITEM_POOL = [
    "Surgical Gloves (Box of 100)", "surgical gloves medium", "SURGICAL GLOVES LARGE",
    "N95 Respirator Masks", "n95 masks - box 20", "Face Masks Surgical",
    "Protective Gowns", "disposable gowns xl", "Isolation Gowns",
    "IV Bags 500ml", "iv bags 1L saline", "IV BAGS NORMAL SALINE",
    "Syringes 10ml", "syringes - 5ml", "Syringe 20ml sterile",
    "Gauze Bandages", "sterile gauze pads", "Bandages Elastic",
    "Catheters Foley 16Fr", "foley catheter kit", "CATHETER URINARY",
    "MRI Scanner Consumables", "CT scanner maintenance", "Ultrasound Gel",
    "X-Ray Film Sheets", "xray developer solution", "Imaging Contrast Media",
    "Scalpel Blades #10", "scalpel handle stainless", "Surgical Scissors",
    "Retractor Set Stainless", "forceps surgical", "NEEDLE HOLDER MAYO",
    "Amoxicillin 500mg Capsules", "ibuprofen 400mg tablets", "Paracetamol IV 1g",
    "Metformin 850mg", "Insulin Glargine 100U", "IV ANTIBIOTICS VANCOMYCIN",
    "Disinfectant Spray 500ml", "hand sanitizer 1L", "Isopropyl Alcohol 70%",
    "Office Paper A4 500 sheets", "ballpoint pens box", "Printer Toner HP",
    "Stapler heavy duty", "file folders box of 50", "Whiteboard Markers",
    "Mop Heads (pack of 6)", "floor cleaner 5L", "Trash Bags 50L",
    "Light Bulbs LED 60W", "cleaning cloths", "Paper Towels Industrial",
]

# Introduce formatting inconsistencies into unit prices
def corrupt_unit_price(price):
    r = random.random()
    if r < 0.25:   return f"${price:.2f}"
    elif r < 0.40: return f"  {price:.2f} "
    else:          return str(round(price, 2))

# Build Bronze line-item rows with intentional corruption + nulls + bad quantities
line_rows = []
line_counter = 1
for i in range(1, 501): # each PO gets 1–6 line items
    for _ in range(random.randint(1, 6)):
        qty_raw  = random.randint(1, 200)
        r        = random.random()
        # Introduce quantity issues: zero, negative, or valid
        quantity = 0 if r < 0.03 else (-abs(qty_raw) if r < 0.05 else qty_raw)
        line_rows.append((
            f"LI-{line_counter:05d}",                     # line_id
            f"PO-{i:04d}",                                # foreign key to purchase orders
            None if random.random() < 0.02 else random.choice(ITEM_POOL),  # 2% null descriptions
            quantity,                                     # messy quantity
            corrupt_unit_price(round(random.uniform(1.5, 800), 2)),  # messy price formats
        ))
        line_counter += 1

line_schema = StructType([
    StructField("line_id",          StringType(),  False),
    StructField("po_id",            StringType(),  True),
    StructField("item_description", StringType(),  True),
    StructField("quantity",         IntegerType(), True),
    StructField("unit_price",       StringType(),  True),  # intentionally STRING
])

df_lines_bronze = spark.createDataFrame(line_rows, schema=line_schema)
write_table(df_lines_bronze, "bronze_line_items")
null_report(df_lines_bronze, "bronze_line_items")
df_lines_bronze.show(5, truncate=False)

20:42:09  INFO  Written -> healthcare_procurement.procurement_analytics.bronze_line_items  (1796 rows)



Null report -- bronze_line_items
+-------+-----+----------------+--------+----------+
|line_id|po_id|item_description|quantity|unit_price|
+-------+-----+----------------+--------+----------+
|      0|    0|              36|       0|         0|
+-------+-----+----------------+--------+----------+

+--------+-------+-----------------------+--------+----------+
|line_id |po_id  |item_description       |quantity|unit_price|
+--------+-------+-----------------------+--------+----------+
|LI-00001|PO-0001|cleaning cloths        |39      |  429.40  |
|LI-00002|PO-0001|Face Masks Surgical    |15      |$347.77   |
|LI-00003|PO-0001|Metformin 850mg        |24      |$100.36   |
|LI-00004|PO-0002|xray developer solution|150     |$41.10    |
|LI-00005|PO-0002|IV Bags 500ml          |143     |433.24    |
+--------+-------+-----------------------+--------+----------+
only showing top 5 rows


### Bronze: `bronze_spend_taxonomy`

**Intentional quality issues:**
- ~5% nulls in `subcategory` and `category`
- Some keywords duplicated with different casing
- A few rows intentionally missing to test unmatched joins in Silver

In [0]:
random.seed(13)

taxonomy_clean = [
    ("surgical gloves",    "PPE",              "Medical Supplies"),
    ("n95",                "PPE",              "Medical Supplies"),
    ("face masks",         "PPE",              "Medical Supplies"),
    ("protective gowns",   "PPE",              "Medical Supplies"),
    ("isolation gowns",    "PPE",              "Medical Supplies"),
    ("iv bags",            "Consumables",      "Medical Supplies"),
    ("syringes",           "Consumables",      "Medical Supplies"),
    ("gauze",              "Consumables",      "Medical Supplies"),
    ("bandages",           "Consumables",      "Medical Supplies"),
    ("catheters",          "Consumables",      "Medical Supplies"),
    ("catheter",           "Consumables",      "Medical Supplies"),
    ("mri",                "Imaging",          "Medical Equipment"),
    ("ct scanner",         "Imaging",          "Medical Equipment"),
    ("ultrasound",         "Imaging",          "Medical Equipment"),
    ("x-ray",              "Imaging",          "Medical Equipment"),
    ("xray",               "Imaging",          "Medical Equipment"),
    ("imaging",            "Imaging",          "Medical Equipment"),
    ("scalpel",            "Surgical Tools",   "Medical Equipment"),
    ("surgical scissors",  "Surgical Tools",   "Medical Equipment"),
    ("retractor",          "Surgical Tools",   "Medical Equipment"),
    ("forceps",            "Surgical Tools",   "Medical Equipment"),
    ("needle holder",      "Surgical Tools",   "Medical Equipment"),
    ("amoxicillin",        "Medications",      "Pharmacy"),
    ("ibuprofen",          "Medications",      "Pharmacy"),
    ("paracetamol",        "Medications",      "Pharmacy"),
    ("metformin",          "Medications",      "Pharmacy"),
    ("insulin",            "Medications",      "Pharmacy"),
    ("antibiotics",        "Medications",      "Pharmacy"),
    ("disinfectant",       "Cleaning Supplies","Facilities"),
    ("hand sanitizer",     "Cleaning Supplies","Facilities"),
    ("isopropyl alcohol",  "Cleaning Supplies","Facilities"),
    ("mop",                "Cleaning Supplies","Facilities"),
    ("floor cleaner",      "Cleaning Supplies","Facilities"),
    ("trash bags",         "Cleaning Supplies","Facilities"),
    ("cleaning cloths",    "Cleaning Supplies","Facilities"),
    ("paper towels",       "Cleaning Supplies","Facilities"),
    ("light bulbs",        "Maintenance",      "Facilities"),
    ("office paper",       "Stationery",       "Administrative"),
    ("ballpoint pens",     "Stationery",       "Administrative"),
    ("printer toner",      "Office Equipment", "Administrative"),
    ("stapler",            "Stationery",       "Administrative"),
    ("file folders",       "Stationery",       "Administrative"),
    ("whiteboard markers", "Stationery",       "Administrative"),
]

# Build Bronze taxonomy rows with intentional casing issues + null injection
taxonomy_rows = []
for idx, (kw, subcat, cat) in enumerate(taxonomy_clean, start=1):
    r    = random.random()

    # Introduce keyword inconsistencies (upper/title/original)
    kw_v = kw.upper() if r < 0.10 else (kw.title() if r < 0.20 else kw)
    taxonomy_rows.append((
        f"TAX-{idx:03d}",                         # taxonomy_id
        kw_v,                                     # messy keyword
        None if random.random() < 0.05 else subcat,  # 5% null subcategory
        None if random.random() < 0.05 else cat,     # 5% null category
    ))
    
# Intentional duplicates with casing issues
taxonomy_rows.append(("TAX-044", "Surgical Gloves", "PPE",         "Medical Supplies"))
taxonomy_rows.append(("TAX-045", "IV BAGS",         "Consumables", None))
taxonomy_rows.append(("TAX-046", "DISINFECTANT",    None,          "Facilities"))

taxonomy_schema = StructType([
    StructField("taxonomy_id",  StringType(), False),
    StructField("keyword",      StringType(), True),
    StructField("subcategory",  StringType(), True),
    StructField("category",     StringType(), True),
])

df_taxonomy_bronze = spark.createDataFrame(taxonomy_rows, schema=taxonomy_schema)
write_table(df_taxonomy_bronze, "bronze_spend_taxonomy")
null_report(df_taxonomy_bronze, "bronze_spend_taxonomy")
df_taxonomy_bronze.show(10, truncate=False)

20:42:13  INFO  Written -> healthcare_procurement.procurement_analytics.bronze_spend_taxonomy  (46 rows)



Null report -- bronze_spend_taxonomy
+-----------+-------+-----------+--------+
|taxonomy_id|keyword|subcategory|category|
+-----------+-------+-----------+--------+
|          0|      0|          4|       2|
+-----------+-------+-----------+--------+

+-----------+----------------+-----------+----------------+
|taxonomy_id|keyword         |subcategory|category        |
+-----------+----------------+-----------+----------------+
|TAX-001    |surgical gloves |PPE        |Medical Supplies|
|TAX-002    |n95             |PPE        |Medical Supplies|
|TAX-003    |Face Masks      |PPE        |Medical Supplies|
|TAX-004    |Protective Gowns|PPE        |Medical Supplies|
|TAX-005    |isolation gowns |PPE        |Medical Supplies|
|TAX-006    |iv bags         |NULL       |Medical Supplies|
|TAX-007    |Syringes        |Consumables|Medical Supplies|
|TAX-008    |gauze           |Consumables|Medical Supplies|
|TAX-009    |bandages        |Consumables|Medical Supplies|
|TAX-010    |catheters    

### Bronze: Validation Summary

In [0]:
summary = []
for tbl in ["bronze_vendors", "bronze_purchase_orders",
            "bronze_line_items", "bronze_spend_taxonomy"]:
    df    = spark.table(tbl)
    cnt   = df.count()
    nulls = {c: df.filter(F.col(c).isNull()).count() for c in df.columns}
    null_summary = ", ".join(f"{c}={n}" for c, n in nulls.items() if n > 0)
    summary.append((tbl, cnt, null_summary or "none detected"))

display(spark.createDataFrame(summary, ["table_name", "row_count", "null_columns"]))

table_name,row_count,null_columns
bronze_vendors,20,city=4
bronze_purchase_orders,500,"paid_date=149, vendor_id=19"
bronze_line_items,1796,item_description=36
bronze_spend_taxonomy,46,"subcategory=4, category=2"


### Define PK / FK Constraints for Catalog Explorer ERD

Adds primary key and foreign key constraints to the Bronze tables so that
Databricks Catalog Explorer can render the Entity Relationship Diagram (ERD).

Note: Databricks enforces `NOT NULL` constraints but treats FK constraints as **informational only**, they define the ERD without blocking bad data. `spend_taxonomy` has no FK since the taxonomy join is a keyword match, not a true FK relationship.

In [0]:
# Define Primary and Foreign Key constraints for Unity Catalog ERD
# Adds PK and FK constraints so Databricks Catalog Explorer can render the ERD.
# Wrapped in try/except — safe to re-run even if constraints already exist.

pk_stmts = [
    # ── Primary keys ──────────────────────────────────────────────────────────
    f"ALTER TABLE {CATALOG}.{SCHEMA}.bronze_vendors           ALTER COLUMN vendor_id   SET NOT NULL",
    f"ALTER TABLE {CATALOG}.{SCHEMA}.bronze_purchase_orders   ALTER COLUMN po_id       SET NOT NULL",
    f"ALTER TABLE {CATALOG}.{SCHEMA}.bronze_line_items        ALTER COLUMN line_id     SET NOT NULL",
    f"ALTER TABLE {CATALOG}.{SCHEMA}.bronze_spend_taxonomy    ALTER COLUMN taxonomy_id SET NOT NULL",

    f"""ALTER TABLE {CATALOG}.{SCHEMA}.bronze_vendors
        ADD CONSTRAINT pk_vendor_id
        PRIMARY KEY (vendor_id)""",

    f"""ALTER TABLE {CATALOG}.{SCHEMA}.bronze_purchase_orders
        ADD CONSTRAINT pk_purchase_orders_id
        PRIMARY KEY (po_id)""",

    f"""ALTER TABLE {CATALOG}.{SCHEMA}.bronze_line_items
        ADD CONSTRAINT pk_line_items_id
        PRIMARY KEY (line_id)""",

    f"""ALTER TABLE {CATALOG}.{SCHEMA}.bronze_spend_taxonomy
        ADD CONSTRAINT pk_spend_taxonomy_id
        PRIMARY KEY (taxonomy_id)""",
]

fk_stmts = [
    # purchase_orders.vendor_id -> vendors.vendor_id
    f"""ALTER TABLE {CATALOG}.{SCHEMA}.bronze_purchase_orders
        ADD CONSTRAINT fk_po_vendor_id
        FOREIGN KEY (vendor_id) REFERENCES {CATALOG}.{SCHEMA}.bronze_vendors(vendor_id)""",

    # line_items.po_id -> purchase_orders.po_id
    f"""ALTER TABLE {CATALOG}.{SCHEMA}.bronze_line_items
        ADD CONSTRAINT fk_li_po_id
        FOREIGN KEY (po_id) REFERENCES {CATALOG}.{SCHEMA}.bronze_purchase_orders(po_id)""",
]

for sql in pk_stmts + fk_stmts:
    try:
        spark.sql(sql)
        log.info("Applied : %s ...", sql.strip()[:70])
    except Exception as e:
        log.warning("Skipped (may already exist): %s", str(e)[:100])

log.info("All PK/FK constraints processed.")
log.info("ERD path: Catalog > %s > %s > [table] > Columns tab > View relationships", CATALOG, SCHEMA)

20:42:29  INFO  Received command c on object id p0
20:42:30  INFO  Applied : ALTER TABLE healthcare_procurement.procurement_analytics.bronze_vendor ...
20:42:31  INFO  Applied : ALTER TABLE healthcare_procurement.procurement_analytics.bronze_purcha ...
20:42:32  INFO  Applied : ALTER TABLE healthcare_procurement.procurement_analytics.bronze_line_i ...
20:42:34  INFO  Applied : ALTER TABLE healthcare_procurement.procurement_analytics.bronze_spend_ ...
20:42:34  INFO  Applied : ALTER TABLE healthcare_procurement.procurement_analytics.bronze_vendor ...
20:42:35  INFO  Applied : ALTER TABLE healthcare_procurement.procurement_analytics.bronze_purcha ...
20:42:36  INFO  Applied : ALTER TABLE healthcare_procurement.procurement_analytics.bronze_line_i ...
20:42:36  INFO  Applied : ALTER TABLE healthcare_procurement.procurement_analytics.bronze_spend_ ...
20:42:37  INFO  Applied : ALTER TABLE healthcare_procurement.procurement_analytics.bronze_purcha ...
20:42:38  INFO  Applied : ALTER TABLE he

### Verify Constraints + ERD Preview

Confirm all 6 constraints (4 PKs + 2 FKs) were applied, then preview
the joined tables to confirm the relationships are working correctly.

In [0]:
# Verify constraints exist in Unity Catalog
spark.sql(f"""
    SELECT constraint_name, constraint_type, table_name
    FROM information_schema.table_constraints
    WHERE table_catalog = '{CATALOG}'
    AND   table_schema  = '{SCHEMA}'
    ORDER BY table_name, constraint_type
""").show(truncate=False)

+---------------------+---------------+----------------------+
|constraint_name      |constraint_type|table_name            |
+---------------------+---------------+----------------------+
|fk_li_po_id          |FOREIGN KEY    |bronze_line_items     |
|pk_line_items_id     |PRIMARY KEY    |bronze_line_items     |
|fk_po_vendor_id      |FOREIGN KEY    |bronze_purchase_orders|
|pk_purchase_orders_id|PRIMARY KEY    |bronze_purchase_orders|
|pk_spend_taxonomy_id |PRIMARY KEY    |bronze_spend_taxonomy |
|pk_vendor_id         |PRIMARY KEY    |bronze_vendors        |
+---------------------+---------------+----------------------+



### Relationship Join Preview

Quick sanity check: joins the three FK-linked tables to confirm
the relationships return the expected data before viewing the ERD.

In [0]:
# Join preview: purchase_orders -> vendors + line_items
df = (
    spark.table(f"{CATALOG}.{SCHEMA}.bronze_purchase_orders").alias("p")
    .join(spark.table(f"{CATALOG}.{SCHEMA}.bronze_line_items").alias("l"),  "po_id",     "left")
    .join(spark.table(f"{CATALOG}.{SCHEMA}.bronze_vendors").alias("v"),     "vendor_id", "left")
)

display(
    df.select(
        "p.po_id",
        "p.vendor_id",
        "v.vendor_name",
        "l.line_id",
        "l.item_description",
        "l.quantity",
        "l.unit_price",
    ).limit(20)
)

po_id,vendor_id,vendor_name,line_id,item_description,quantity,unit_price
PO-0001,V017,Cintas Corporation,LI-00003,Metformin 850mg,24,$100.36
PO-0002,V007,bd (becton dickinson),LI-00009,Gauze Bandages,64,66.86
PO-0003,V014,Olympus Medical,LI-00013,n95 masks - box 20,153,672.21
PO-0004,V015,Zimmer Biomet,LI-00017,forceps surgical,34,65.84
PO-0005,V020,ge healthcare,LI-00022,Disinfectant Spray 500ml,33,524.49
PO-0006,V016,Staples Business Advantage,LI-00026,n95 masks - box 20,26,699.65
PO-0007,V007,bd (becton dickinson),LI-00028,Retractor Set Stainless,125,$250.51
PO-0008,null,null,LI-00034,forceps surgical,190,$285.42
PO-0009,V008,Siemens Healthineers,LI-00035,IV ANTIBIOTICS VANCOMYCIN,72,765.28
PO-0010,V003,johnson & johnson,LI-00041,Isolation Gowns,22,23.5
